<a id="top"></a>

# Lab L0806: write the description, design the schema

```yaml
title: "Lab L0806: write the description, design the schema"
keywords: tool description, rich description, parameter schema, enum, per-field description, required, validate arguments, lab
estimated duration: 30
```

> **Lesson:** L08. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L08/objectives.md). Solutions: `L0806_lab_solutions.ipynb`. Pairs with the [L0805 demo](L0805_lecture.ipynb).
>
> **Pure Python — no API key needed.** L08 is about *design judgment*, and you can practice that offline: you design schemas, rewrite errors, and classify tasks without calling a model. (The teacher demos make the live calls; these labs make the design choices.)
>
> **After this lab you can:** rewrite a sparse description into one written *for the model*, design a tight parameter schema (types, required, enum, per-field constraints), and validate sample arguments against it — all in pure Python.

## Contents

- [1. Setup — a sparse tool to improve](#1-setup--a-sparse-tool-to-improve)
- [2. Problem 1 — Rewrite the description for the model](#2-problem-1--rewrite-the-description-for-the-model)
- [3. Problem 2 — Tighten the parameter schema](#3-problem-2--tighten-the-parameter-schema)
- [4. Problem 3 — Validate sample arguments against your schema](#4-problem-3--validate-sample-arguments-against-your-schema)
- [5. Problem 5 — Why an enum over a free string? (written)](#5-problem-5--why-an-enum-over-a-free-string-written)
- [6. Self-check](#6-self-check)

## 1. Setup — a sparse tool to improve

A `set_priority(ticket_id, level)` tool with a **sparse** description and a **loose** schema. Your job across the lab is to make it a tool a model can use correctly on first read.

In [1]:
SPARSE_TOOL: dict[str, object] = {
    "name": "set_priority",
    "description": "Sets priority.",  # sparse: no when/what/return guidance
    "input_schema": {
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string"},
            "level": {"type": "string"},  # loose: any string accepted
        },
        "required": ["ticket_id"],  # loose: level not even required
    },
}
print("sparse tool loaded:", SPARSE_TOOL["name"])

sparse tool loaded: set_priority


[↑ Back to top](#top)

## 2. Problem 1 — Rewrite the description for the model

Write `RICH_DESCRIPTION`: a 3–5 sentence string that answers all four questions from the lecture — *what it does*, *when to call it*, *when NOT to call it*, *what it returns*. Include the accepted `level` values and a one-line example. The check below asserts your description mentions the key facts; the *wording* is yours.

In [2]:
RICH_DESCRIPTION = (
    "Set the priority of an existing support ticket. Call this when the user explicitly "
    "asks to change a ticket's priority and names (or has named) the ticket. Do NOT call it "
    "to create a ticket or to read its current priority — it only updates an existing one. "
    "The level must be one of 'low', 'medium', or 'high'. Returns "
    "{'ticket_id', 'level', 'updated': true} on success, e.g. "
    "{'ticket_id': 'T-42', 'level': 'high', 'updated': true}."
)

# Lightweight check that the description covers the essentials (not graded on wording):
low = RICH_DESCRIPTION.lower()
for needle in ["ticket", "low", "high", "return"]:
    print(f"mentions {needle!r}:", needle in low)
assert len(RICH_DESCRIPTION.split()) >= 25, "aim for 3-5 real sentences"

mentions 'ticket': True
mentions 'low': True
mentions 'high': True
mentions 'return': True


[↑ Back to top](#top)

## 3. Problem 2 — Tighten the parameter schema

Build `TIGHT_SCHEMA` (the JSON-Schema `input_schema` dict): both `ticket_id` and `level` **required**; `level` an **enum** of `low`/`medium`/`high`; and a **per-field description** on each property (each should carry an example or a constraint, per the lecture's rule of thumb).

In [3]:
TIGHT_SCHEMA: dict[str, object] = {
    "type": "object",
    "properties": {
        "ticket_id": {
            "type": "string",
            "description": "the ticket id to update, e.g. 'T-42'.",
        },
        "level": {
            "type": "string",
            "enum": ["low", "medium", "high"],
            "description": "the new priority; one of low, medium, high.",
        },
    },
    "required": ["ticket_id", "level"],
}

[↑ Back to top](#top)

## 4. Problem 3 — Validate sample arguments against your schema

Implement `validate(args, schema)` (a small subset of JSON-Schema validation): every `required` key present; each present value matches its property `type` (`string`->`str`, `integer`->`int`); and if a property has an `enum`, the value is in it. Return `"ok"` or a short reason string. Run it over the sample calls.

In [4]:
SAMPLES = [
    {"ticket_id": "T-42", "level": "high"},  # ok
    {"ticket_id": "T-7"},  # missing required level
    {"ticket_id": "T-9", "level": "urgent"},  # not in enum
    {"ticket_id": 9, "level": "low"},  # wrong type for ticket_id
]


TYPES = {"string": str, "integer": int, "boolean": bool}


def validate(args: dict[str, object], schema: dict[str, object]) -> str:
    """Tiny JSON-Schema check: required present, types match, enum membership."""
    props = schema["properties"]
    for key in schema.get("required", []):
        if key not in args:
            return f"missing required field {key!r}"
    for key, value in args.items():
        spec = props.get(key)
        if spec is None:
            return f"unknown field {key!r}"
        expected = TYPES[spec["type"]]
        # bool is a subclass of int in Python; guard it out for integer fields.
        if not isinstance(value, expected) or (expected is int and isinstance(value, bool)):
            return f"field {key!r} must be {spec['type']}, got {type(value).__name__}"
        if "enum" in spec and value not in spec["enum"]:
            return f"field {key!r} must be one of {spec['enum']}, got {value!r}"
    return "ok"


for s in SAMPLES:
    print(s, "->", validate(s, TIGHT_SCHEMA))

{'ticket_id': 'T-42', 'level': 'high'} -> ok
{'ticket_id': 'T-7'} -> missing required field 'level'
{'ticket_id': 'T-9', 'level': 'urgent'} -> field 'level' must be one of ['low', 'medium', 'high'], got 'urgent'
{'ticket_id': 9, 'level': 'low'} -> field 'ticket_id' must be string, got int


[↑ Back to top](#top)

## 5. Problem 5 — Why an enum over a free string? (written)

Your tight schema makes `level` an enum instead of a free-form string. In a sentence or two: what does the enum buy you in terms of *model behavior* and *validation* (think about the L0805 demo's sparse-vs-rich contrast)?

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0806_lab_solutions.ipynb`. Done when your `RICH_DESCRIPTION` answers all four questions, `TIGHT_SCHEMA` makes both fields required with a `level` enum, and `validate` returns `ok` for the first sample and a useful reason for the other three.

[↑ Back to top](#top)